In [0]:
import uuid
from datetime import datetime, timezone

PIPELINE_NAME = "bcb_sgs_selic_pipeline"
SOURCE_SYSTEM = "BCB_SGS"
DATASET_NAME = "bcb_sgs_selic"

execution_id = str(uuid.uuid4())
pipeline_start_timestamp = datetime.now(timezone.utc)

print("Execution ID:", execution_id)
print("Pipeline start:", pipeline_start_timestamp)

In [0]:
spark.sql(f"""
INSERT INTO workspace.brazilian_economic_monitoring.pipeline_execution
(
    execution_id,
    pipeline_name,
    source_system,
    dataset_name,
    start_timestamp,
    end_timestamp,
    duration_seconds,
    status,
    error_message
)
VALUES
(
    '{execution_id}',
    '{PIPELINE_NAME}',
    '{SOURCE_SYSTEM}',
    '{DATASET_NAME}',
    CURRENT_TIMESTAMP(),
    NULL,
    NULL,
    'RUNNING',
    NULL
)
""")

In [0]:
steps = [
    ("INGEST_BRONZE", 1, "INGESTION", "BRONZE"),
    ("DQ_BRONZE", 2, "DATA_QUALITY", "GOVERNANCE"),
    ("LOAD_SILVER", 3, "TRANSFORMATION", "SILVER"),
    ("DQ_SILVER", 4, "DATA_QUALITY", "GOVERNANCE"),
    ("LOAD_GOLD", 5, "ANALYTICAL", "GOLD")
]

for step_name, step_order, step_type, layer in steps:
    spark.sql(f"""
        INSERT INTO workspace.brazilian_economic_monitoring.pipeline_step_execution
        (
            execution_id,
            step_name,
            step_order,
            step_type,
            layer,
            start_timestamp,
            end_timestamp,
            duration_seconds,
            status,
            records_read,
            records_inserted,
            records_updated,
            records_output,
            failed_checks,
            error_message
        )
        VALUES
        (
            '{execution_id}',
            '{step_name}',
            {step_order},
            '{step_type}',
            '{layer}',
            NULL,
            NULL,
            NULL,
            'PENDING',
            NULL,
            NULL,
            NULL,
            NULL,
            NULL,
            NULL
        )
    """)

In [0]:
dbutils.jobs.taskValues.set(
    key="execution_id",
    value=execution_id
)